In [ ]:
import openai
import pinecone
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

In [ ]:
# Read doc
def read_doc(directory):
  file_loader = PyPDFDirectoryLoader(directory)
  documents= file_loader.load()
  return documents

In [ ]:
# directory name in which the pdfs have been stored
docs = read_doc("acts/")
len(docs)

In [ ]:
## Divide the docs into chunks
def chunk_data(docs,chunk_size= 800,chunk_overlap = 50):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  docs = text_splitter.split_documents(docs)
  return docs

In [ ]:
documents = chunk_data(docs=docs)
len(documents)

In [ ]:
#embedding techinques of OpenAI
embeddings= OpenAIEmbeddings(api_key = os.getenv("OPENAI_API_KEY"))
embeddings

In [ ]:
pc = pinecone.Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name= os.getenv("PINECONE_INDEX_NAME")

In [ ]:
pc.list_indexes()

In [ ]:

index = pc.Index(index_name)

In [ ]:
index.describe_index_stats()

In [ ]:
import tqdm

In [ ]:

vectors = []
for i, doc in tqdm.tqdm(enumerate(documents)):
    embedding = embeddings.embed_query(doc.page_content)  # Generate embedding
    vectors.append((f"doc-{i}", embedding, {"text": doc.page_content}))  # Format



In [ ]:
# so now we have to upsert the vectors batch wise
# to avoid the memory issues
batch_size = 200
for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i+batch_size]
    index.upsert(batch)

In [ ]:
index.describe_index_stats()

In [ ]:
# Everything after this is its RAG implementation

PROMPT_QUESTION_TEMPLATE = "You are a genius chatbot and can answer questions related to law given that you are provided with the text that contains the answer. Please answer the following question:\n\n"
PROMPT_ANSWER_TEMPLATE = "\n\n The answer to the provided questions is inside the following text:\n\n"

def generate_prompt(prompt,answer):
    return PROMPT_QUESTION_TEMPLATE + prompt + PROMPT_ANSWER_TEMPLATE + answer


In [ ]:
def reply(prompt):
    res = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a genius chatbot and can answer questions related to law given that you are provided with the text that contains the answer."},
            {"role": "user", "content": prompt}
        ]
    )
    return res.choices[0].message.content

In [ ]:
def chat(query):
  embed_model = 'text-embedding-ada-002'
  res = openai.embeddings.create(
    input= [query],
    model=embed_model
  )
  xq= res.data[0].embedding

  res = index.query(vector=xq, top_k=4,include_metadata=True)
  answer = ""
  for result in res["matches"]:
    answer += result["metadata"]["text"]
  return reply(generate_prompt(query,answer))

In [ ]:
print(chat(query="Can I do suicide?"))